In [1]:
# Library Imports
import pandas as pd
from pyld import jsonld
import json
from pathlib import Path 
import re
from ydata_profiling import ProfileReport

In [2]:
gene_graph_files_directory = Path("genegraph_data/gene-validity-jsonld-dec-2024//") # Directory containing data files
gene_graph_files = list(gene_graph_files_directory.glob("*.json"))
len(gene_graph_files)


2913

#### JSON-LD Frames  Parsing
- Purpose to extract entities from the files

##### Issues:
- Need to merge the frames, i.e get one code for both the frames
- Some issue while framing the gene, diseas data

In [3]:
gene_frame = {"@context": {
    "@vocab": "http://dataexchange.clinicalgenome.org/terms/",
    "id": "@id",
    "type": "@type",
    "cg": "http://dataexchange.clinicalgenome.org/terms/",
    "dc": "http://purl.org/dc/terms/",
    "obo": "http://purl.obolibrary.org/obo/",
    "hgnc": "https://identifiers.org/hgnc:",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "cggv": "http://dataexchange.clinicalgenome.org/gci/",
    "evidenceStrength": {
      "@type": "@vocab"
    },
    "calculatedEvidenceStrength": {
      "@type": "@vocab"
    },
    "computedEvidenceStrength": {
      "@type": "@vocab"
    },
    "specifiedBy": {
      "@type": "@vocab"
    },
    "evidence": {
      "@container": "@set"
    },
    "disease": {
      "@type": "@vocab"
    },
    "gene": {
      "@type": "@vocab"
    },
    "phenotypes": {
      "@type": "@id"
    },
    "agent": {
      "@type": "@id"
    },
    "role": {
      "@type": "@vocab"
    },
    "modeOfInheritance": {
      "@type": "@vocab"
    },
    "sex": {
      "@type": "@vocab"
    },
    "direction": {
      "@type": "@vocab"
    },
    "ageType": {
      "@type": "@vocab"
    },
    "ageUnit": {
      "@type": "@vocab"
    },
    "dc:source": {
      "@type": "@id"
    }
  },
"gene":{}

}


In [4]:
proband_frame = {"@context": {
    "@vocab": "http://dataexchange.clinicalgenome.org/terms/",
    "id": "@id",
    "type": "@type",
    "cg": "http://dataexchange.clinicalgenome.org/terms/",
    "dc": "http://purl.org/dc/terms/",
    "obo": "http://purl.obolibrary.org/obo/",
    "hgnc": "https://identifiers.org/hgnc:",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "cggv": "http://dataexchange.clinicalgenome.org/gci/",
    "evidenceStrength": {
      "@type": "@vocab"
    },
    "calculatedEvidenceStrength": {
      "@type": "@vocab"
    },
    "computedEvidenceStrength": {
      "@type": "@vocab"
    },
    "specifiedBy": {
      "@type": "@vocab"
    },
    "evidence": {
      "@container": "@set"
    },
    "disease": {
      "@type": "@vocab"
    },
    "gene": {
      "@type": "@vocab"
    },
    "phenotypes": {
      "@type": "@id"
    },
    "agent": {
      "@type": "@id"
    },
    "role": {
      "@type": "@vocab"
    },
    "modeOfInheritance": {
      "@type": "@vocab"
    },
    "sex": {
      "@type": "@vocab"
    },
    "direction": {
      "@type": "@vocab"
    },
    "ageType": {
      "@type": "@vocab"
    },
    "ageUnit": {
      "@type": "@vocab"
    },
    "dc:source": {
      "@type": "@id"
    }
  },
 "type":"Proband"
}


#### Parsing 

In [5]:
entities_data = []
pubmed_id_regex = r"https://pubmed.ncbi.nlm.nih.gov/(\d+)"
for gene_graph_file in gene_graph_files :
    # print(gene_graph_file.name)
    with open(gene_graph_file,'r') as fh:
        data = json.load(fh)
        try:
            gene_data = jsonld.frame(data, gene_frame)
            # print(f"gene:{gene_data['gene']}")
            # print(f"disease:{gene_data['disease']}")
            # print(f"modeOfInheritance:{gene_data['modeOfInheritance']}")
            try:
                proband_data = jsonld.frame(data, proband_frame)
                graph_data = proband_data["@graph"]
                for i, items in enumerate(graph_data):
                    # print(items)
                    errors = []
                    # print(f"Proband # {i+1}")
                    # print(items)
                    # print(f"id:{items['id']}")
                    try:
                        items.get('phenotypes')
                        phenotypes = items.get('phenotypes')
                        number_of_phenotypic_terms = len(phenotypes)
                    except:
                        phenotypes = []
                        number_of_phenotypic_terms = 0
                        errors.append("phenotypes key not found in the proband dictionary") 
                    try :
                        variant_info = items.get('variant')
                        pubmed_id = re.search(pubmed_id_regex, str(variant_info)).groups()[0] 
                    except:
                        errors.append(" variant key not found in the proband dictionary, thus no pubmed_id")
                    # print(f"phenotypes:{phenotypes}")
                    # print(f"number_of_phenotypic_terms:{number_of_phenotypic_terms}")
                    # print(f"pubmed_id:{pubmed_id}") 
                    entities_data.append({"file":gene_graph_file.name,
                                         "gene":gene_data['gene'],
                                         "disease":gene_data['disease'],
                                         "modeOfInheritance":gene_data['modeOfInheritance'],
                                         "Proband #":f"{i+1}", 
                                         "proband_id":items['id'],
                                         "phenotypes":phenotypes,
                                         "number_of_phenotypic_terms":number_of_phenotypic_terms,
                                         "pubmed_id":pubmed_id,
                                          "errors":errors})
                    
            except:
                entities_data.append({"file":gene_graph_file.name,
                                         "gene":gene_data['gene'],
                                         "disease":gene_data['disease'],
                                         "modeOfInheritance":gene_data['modeOfInheritance'],
                                         "Proband #":None, 
                                         "proband_id":None,
                                         "phenotypes":None,
                                         "number_of_phenotypic_terms":None,
                                         "pubmed_id":None,
                                          "errors": ['Error in fetching proband frame in the file']})       
        except:
            print(f"Error in fetchinge gene framing in the file at location {gene_graph_file}")
print("Code Over")     

Code Over


In [7]:
entities = pd.DataFrame(entities_data)

In [8]:

entities.to_csv("extracted_entities.csv", index=False)

#### EDA

In [38]:
entities = pd.read_csv("extracted_entities.csv")
entities.head()

,file,gene,disease,modeOfInheritance,Proband #,proband_id,phenotypes,number_of_phenotypic_terms,pubmed_id,errors
0,cggv_0c056edd-f72b-43e9-ba40-7504f285cf5av1.0....,hgnc:25941,obo:MONDO_0015924,obo:HP_0000006,NaN,NaN,NaN,NaN,NaN,['Error in fetching proband frame in the file']
1,cggv_9153b41e-f2ef-453d-883b-187565cf4593v1.1....,hgnc:2336,obo:MONDO_0013862,obo:HP_0000007,1.0,cggv:38a5aad0-42c4-4c38-9dd1-03ba1df17d41,"['obo:HP_0030388', 'obo:HP_0002014', 'obo:HP_0...",8.0,22035880.0,[]
2,cggv_9153b41e-f2ef-453d-883b-187565cf4593v1.1....,hgnc:2336,obo:MONDO_0013862,obo:HP_0000007,2.0,cggv:54dddeee-e1d2-45b4-9120-9914dbc79e93,"['obo:HP_0004313', 'obo:HP_0030374', 'obo:HP_0...",9.0,26325596.0,[]
3,cggv_9153b41e-f2ef-453d-883b-187565cf4593v1.1....,hgnc:2336,obo:MONDO_0013862,obo:HP_0000007,3.0,cggv:aef633a7-7ac8-4dbc-8914-ab2cc11296b4,"['obo:HP_0000405', 'obo:HP_0002090', 'obo:HP_0...",8.0,28499783.0,[]
4,cggv_659c7312-5d0a-44d0-9bd0-3e2b2ba385cev1.2....,hgnc:28957,obo:MONDO_0100516,obo:HP_0000007,1.0,cggv:616c46be-04a4-4f07-90b8-64cc66d7ae29,"['obo:HP_0002451', 'obo:HP_0025190', 'obo:HP_0...",15.0,29271071.0,[]


#### Data conversion for better EDA

In [40]:
entities['number_of_phenotypic_terms'] = entities['number_of_phenotypic_terms'].fillna(0)
entities['number_of_phenotypic_terms'] = entities['number_of_phenotypic_terms'].astype(int)

In [41]:
entities['Proband #'] = entities['Proband #'].fillna(0)
entities['Proband #'] = entities['Proband #'].astype(int)

In [42]:
entities['pubmed_id'] = entities['pubmed_id'].astype(str)
entities['pubmed_id'] = entities['pubmed_id'].map( lambda x : x.split('.')[0])

In [48]:
profile = ProfileReport(entities, title ='Basic EDA on Genegraph Data')

In [49]:
profile.to_file('genegraph_eda_ydata.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


%|                                                    | 0/10 [00:00<?, ?it/s]
%|████▍                                       | 1/10 [00:00<00:01,  8.80it/s]
100%|███████████████████████████████████████████| 10/10 [00:00<00:00, 38.17it/s]
/opt/anaconda3/envs/ra_sandbox/lib/python3.12/site-packages/matplotlib/axes/_axes.py:5139: RuntimeWarning: invalid value encountered in cast
  ix1 = np.round(ix).astype(int)
/opt/anaconda3/envs/ra_sandbox/lib/python3.12/site-packages/matplotlib/axes/_axes.py:5141: RuntimeWarning: invalid value encountered in cast
  ix2 = np.floor(ix).astype(int)
/opt/anaconda3/envs/ra_sandbox/lib/python3.12/site-packages/matplotlib/axes/_axes.py:5139: RuntimeWarning: invalid value encountered in cast
  ix1 = np.round(ix).astype(int)
/opt/anaconda3/envs/ra_sandbox/lib/python3.12/site-packages/matplotlib/axes/_axes.py:5141: RuntimeWarning: invalid value encountered in cast
  ix2 = np.floor(ix).astype(int)
/opt/anaconda3/envs/ra_sandbox/lib/python3.12/site-packages/matplo

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]